# Text2MQL demo

# Text2MQL demo

In [1]:
from external import TEXT2VQL_ROOT as ROOT #Load text2vql project to system path.

import os

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

from text2vql.seed.seed_yakindu import TEXT2VQL_ROOT
from text2vql.util.metamodel import MetaModel
from templates import COMPLETION_QUERY, QUERY

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

## Load model

Set base_model_id to the open-source model
Set adapter_id to the required fine-tuned model
Set lang to the query language

In [5]:
base_model_id = "codellama/CodeLlama-7b-hf"
adapter_id = "PELAB-LiU/Text2MQL-CodeLlama-7b"
lang = "ocl"

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
base_model = AutoModelForCausalLM.from_pretrained(base_model_id, device_map="auto")

model = PeftModel.from_pretrained(base_model, adapter_id, subfolder=lang)

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

ocl/adapter_model.safetensors:   0%|          | 0.00/33.6M [00:00<?, ?B/s]

## Run demo

Set metamodel to a meta-model
Set instruction to your prompt
Set header to initial hint

In [6]:
metamodel = MetaModel(os.path.join(TEXT2VQL_ROOT,'dataset_construction/seed/yakindu_simplified.ecore'))
instruction = "Find states with at least 2 outgoing transition."
header = "Set<RegularState>"
prompt = COMPLETION_QUERY[lang].safe_substitute(
                    metamodel=metamodel.get_metamodel_info(),
                    description=instruction,
                    header=header
)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=150)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


            
```
abstract class Pseudostate extends Vertex {
}
abstract class Vertex {
	reference Transition[0..*] incomingTransitions;
	reference Transition[0..*] outgoingTransitions;
}
class Region {
	reference Vertex[0..*] vertices;
	attribute EString[0..1] name;
}
class Transition {
	reference Vertex[1..1] target;
	reference Vertex[0..1] source;
}
class Statechart extends CompositeElement {
}
class Entry extends Pseudostate {
}
class Synchronization extends Pseudostate {
}
class State extends RegularState, CompositeElement {
}
abstract class RegularState extends Vertex {
}
abstract class CompositeElement {
	reference Region[0..*] regions;
}
class Choice extends Pseudostate {
}
class Exit extends Pseudostate {
}
class FinalState extends RegularState {
}

```
Find states with at least 2 outgoing transition.
Set<RegularState>
```ocl
RegularState.allInstances()->select(s |
    s.outgoingTransitions->size() >= 2
)
```

